# 00 — Setup, GURU schema audit, memory gate, smoke test

**Project:** RLVR plasticity, exp2 Colab variant (Math -> Simulation, Qwen2.5-7B, LoRA) · **Owner:** Aaron (Person 4) · **Plan:** `EXPERIMENT_2_COLAB_PLAN.md`

Nothing in this notebook trains beyond a 2-update smoke test. Everything here is a discovery step whose output gets committed to `data/`. **Do not run 01 until every gate in this notebook has passed and the schema audit has been manually confirmed** (see the cell below marked MANUAL REVIEW REQUIRED).

**Deviations logged in this notebook:** none — like eaaj-pilot's own 00, this notebook makes no training choices, only discoveries.

In [ ]:
%pip install -q -r "/content/RLVR/experiment 2/requirements.txt"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
REPO_URL = 'https://github.com/WYR186/RLVR.git'  # HTTPS; if the repo is
# private, authenticate interactively (git credential prompt / a token you
# paste when asked) rather than embedding a token in this notebook.
REPO_DIR = '/content/RLVR'
import os
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

EXP2_DIR = f'{REPO_DIR}/experiment 2'
sys.path.insert(0, EXP2_DIR)  # only this one goes on sys.path — pipeline.py
# reaches eaaj-pilot/src by explicit file path internally, avoiding a
# top-level `src` package-name collision between the two sibling dirs
# (see experiment 2/src/pipeline.py's module docstring).

import src.guru_data as guru_data
import src.guru_reward as guru_reward
import src.pipeline as pipeline

import json
from pathlib import Path
CONFIG = json.load(open(f'{EXP2_DIR}/exp2_colab_config.json'))
DATA_DIR = Path(EXP2_DIR) / 'data'
print('config loaded:', CONFIG['experiment'])

## Phase 0 step 1-3 — schema audit + subset filtering + answer-format discovery (heuristic proposal)

In [ ]:
audit = guru_data.audit_schema(revision=CONFIG['dataset'].get('revision'))
print('domain_field_candidate:', audit['domain_field_candidate'])
print('prompt_field_candidate:', audit['prompt_field_candidate'])
print('answer_field_candidate:', audit['answer_field_candidate'])
print('evidence:', audit['domain_field_evidence'])
for name, ex in audit['examples_by_subset'].items():
    print('---', name, '---')
    print(ex['row'])

### MANUAL REVIEW REQUIRED before continuing

Read the printed candidates (domain, prompt, answer fields), the evidence counts, and every example row above. Compare against the actual GURU / guru-RL-92k documentation. Check specifically: (a) the domain field genuinely identifies OR1/DAPO/DeepScaler (Math) and CodeI/O (Simulation) rows; (b) the answer field's *format* matches what `src/guru_reward.py`'s extractors parse (boxed/numeric for Math, normalized output string for CodeI/O) — if not, extend `guru_reward.py` before continuing, do not force a mismatch through. If any candidate is wrong, correct it in the JSON below before setting the confirmed flag — do not confirm a wrong field.

In [ ]:
import json as _json, pathlib as _pathlib
_audit_path = _pathlib.Path(EXP2_DIR) / 'data' / 'guru_schema_audit.json'
_audit = _json.loads(_audit_path.read_text())
# If the review above found a wrong candidate, correct it here first, e.g.:
# _audit['prompt_field_candidate'] = 'the_real_column_name'
_audit['domain_field_manually_confirmed'] = True  # only after the review above
_audit_path.write_text(_json.dumps(_audit, indent=1))
print('confirmed:', _audit['domain_field_manually_confirmed'])

## Phase 0 step 4 — token-length audit (GATE 0a)

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_id'])

# Small provisional samples for the audit only — build_exp2_splits (next
# section) draws the real frozen train/eval sets.
import random
audit_confirmed = guru_data.load_confirmed_audit()
domain_field = audit_confirmed['domain_field_candidate']
PROMPT_FIELD = audit_confirmed['prompt_field_candidate']
ANSWER_FIELD = audit_confirmed['answer_field_candidate']
assert PROMPT_FIELD and ANSWER_FIELD, 'confirm real field names in the audit before continuing'
raw = guru_data._load_raw(revision=audit_confirmed.get('revision_requested'))
train_split = raw['train'] if 'train' in raw else next(iter(raw.values()))
stage_a_pool = guru_data.filter_stage_subset(train_split, guru_data.STAGE_A_SUBSET_NAMES, domain_field)
stage_b_pool = guru_data.filter_stage_subset(train_split, guru_data.STAGE_B_SUBSET_NAMES, domain_field)
print('stage A (Math) pool size:', len(stage_a_pool))
print('stage B (Simulation) pool size:', len(stage_b_pool))

In [ ]:
sample_a = [str(r[PROMPT_FIELD]) for r in stage_a_pool.select(range(min(200, len(stage_a_pool))))]
sample_b = [str(r[PROMPT_FIELD]) for r in stage_b_pool.select(range(min(200, len(stage_b_pool))))]
audit_a = guru_data.token_length_audit(sample_a, tokenizer, 'stage_a_math')
audit_b = guru_data.token_length_audit(sample_b, tokenizer, 'stage_b_simulation')
token_audit = {'stage_a': audit_a, 'stage_b': audit_b}
print(token_audit)
(_pathlib.Path(EXP2_DIR) / 'data' / 'token_length_audit.json').write_text(_json.dumps(token_audit, indent=1))

gate_0a_threshold = CONFIG['gates']['phase0a_stage_b_p95_prompt_tokens_max']
if audit_b['p95'] > gate_0a_threshold:
    raise SystemExit(
        f"GATE 0a STOP: stage-B p95={audit_b['p95']} > {gate_0a_threshold}. "
        'Escalate max_prompt_length / GPU tier — do not shrink the batch to force a fit.')
print('GATE 0a: PASS')

## Phase 0 step 5 — freeze splits (incl. the frozen 4096-prompt Q-metric probe)

In [ ]:
splits = guru_data.build_exp2_splits(
    n_train_a=512, n_train_b=256, n_eval_b=CONFIG['stage_b']['eval_questions'],
    n_probe=CONFIG['measurement']['probe_questions'], seed=CONFIG['seed'])
print('stage_a_train:', len(splits['stage_a_train_idx']),
      'stage_b_train:', len(splits['stage_b_train_idx']),
      'stage_b_eval:', len(splits['stage_b_eval_idx']),
      'probe:', splits['probe_actual'], '/', splits['probe_requested'])
if 'probe_shortfall_note' in splits:
    print('WARNING:', splits['probe_shortfall_note'])

## Gate C0 — GPU memory calibration (escalate L4 -> A100, do not shrink)

In [ ]:
# Tiny throwaway dataset (a handful of stage-A rows mapped to 'prompt'/
# 'answer' columns) for the 2-update smoke run used by both Gate C0 and
# the Phase-0 smoke test below. Field names come from the confirmed audit.
smoke_rows = stage_a_pool.select(range(8))
smoke_ds = smoke_rows.map(lambda ex: {
    'prompt': str(ex[PROMPT_FIELD]), 'answer': str(ex[ANSWER_FIELD])
}, remove_columns=smoke_rows.column_names)

gate_c0 = pipeline.gate_c0_memory_probe(
    CONFIG['model_id'], CONFIG['peft'], smoke_ds, device='cuda',
    min_headroom_pct=CONFIG['gates']['gate_c0_memory_headroom_min_pct'])
print(gate_c0)
if not gate_c0['gate_pass']:
    print('Gate C0: escalate to A100 (switch the Colab runtime, then re-run this cell).')
else:
    print('Gate C0: PASS on current tier.')

## Phase 0 step 7-8 — smoke test + sparse-reward preflight (GATE 0b)

In [ ]:
model, model_tokenizer = pipeline.build_peft_model(
    CONFIG['model_id'], CONFIG['peft'], device='cuda')
eval_prompts_smoke = [r['prompt'] for r in smoke_ds]
eval_golds_smoke = [r['answer'] for r in smoke_ds]
preflight = pipeline.guru_sparse_reward_preflight(
    model, model_tokenizer, eval_prompts_smoke, eval_golds_smoke, 'Math',
    num_generations=CONFIG['stage_a']['num_generations'])
print('has_grpo_signal:', preflight['has_grpo_signal'],
      'groups_with_variance:', preflight['groups_with_reward_variance'], '/', preflight['n_prompts'])
if not preflight['has_grpo_signal']:
    raise SystemExit('GATE 0b STOP: every sampled group has constant reward. '
                     'Preserve this preflight result and ask the team — do not add '
                     'a shaping reward, do not switch models.')
print('GATE 0b: PASS')
del model, model_tokenizer

## Commit reminder

Commit `data/guru_schema_audit.json`, `data/token_length_audit.json`, `data/exp2_splits.json` with message prefix `exp2-colab:`. Log this phase's wall time and Colab compute-unit cost in `eaaj-pilot/compute_log.md` before moving to notebook 01.